# 01_Theory: 토크나이제이션의 3가지 알고리즘

## 개요
이 노트북은 **BPE (Byte Pair Encoding)**, **WordPiece**, **SentencePiece** 3가지 토크나이제이션 알고리즘을 깊이있게 이해하기 위한 자료입니다.

### 핵심 질문
- LLM은 텍스트를 숫자로만 이해합니다
- "machine", "machines", "learning" 같은 단어들을 어떻게 효율적으로 숫자로 변환할까요?
- 각 알고리즘은 어떤 선택과 트레이드오프를 하고 있을까요?

---

## 1. 토크나이제이션의 기본 개념

### 세 가지 단위 선택지

| 단위 | 예시 | 장점 | 단점 |
|------|------|------|------|
| **문자** | ['m','a','c','h','i','n','e'] | 어휘 크기 작음 | 정보 분산, 비효율 |
| **단어** | ['machine'] | 직관적 | 어휘 폭발, 변형 처리 어려움 |
| **서브워드** | ['mach','ine'] | 효율과 직관성 균형 | 알고리즘 복잡 |

### 언어별 토크나이제이션 효율성

```
영어:   "hello world" → 2개 토큰 (효율적)
한글:   "안녕하세요" → 2개 토큰 (보통)
중국어: "你好"      → 2개 토큰 (비효율적)

→ 같은 의미도 언어마다 토큰 소모가 다름!
→ LLM 비용과 성능에 직접 영향
```

---

## 2. BPE (Byte Pair Encoding) - GPT-2

### 2.1 핵심 아이디어
**"가장 자주 나타나는 인접한 두 개의 토큰을 계속 병합한다"**

### 2.2 알고리즘 흐름

#### Step 1: 초기 상태 - 모든 문자를 개별 토큰으로
```
"hello world" → ['h','e','l','l','o',' ','w','o','r','l','d']
```

#### Step 2: 가장 빈번한 인접 쌍 찾기
```
('h','e'): 빈도 3
('l','l'): 빈도 5  ← 가장 빈번!
('o',' '): 빈도 2
```

#### Step 3: 해당 쌍 병합
```
Before: ['h','e','l','l','o',' ','w','o','r','l','d']
After:  ['h','e','ll','o',' ','w','o','r','l','d']
```

#### Step 4: 반복 (목표 어휘 크기까지)
```
다음: ('e','ll') 병합
그 다음: ('ell','o') 병합
...
반복 (약 50,000번) → GPT-2 어휘 크기
```

### 2.3 특징
- **선택 기준**: 빈도 (Greedy 접근)
- **공백 처리**: 특수 문자로 표현 (예: `hello</w>world`)
- **사용 모델**: GPT-2, GPT-3, GPT-4
- **어휘 크기**: ~50,000개

---

## 3. WordPiece - BERT

### 3.1 핵심 아이디어
**"병합했을 때 모델 우도(Likelihood)가 가장 증가하는 쌍을 선택한다"**

### 3.2 핵심 개념: 정보 이득(Information Gain)

```
정보 이득 = log(P(AB) / (P(A) × P(B)))

설명:
- P(AB) = 병합된 토큰의 확률
- P(A) × P(B) = 각각 따로 나타날 확률

이득이 크다 = 이 두 문자가 자주 함께 나타난다
           = 함께 병합하는 것이 의미있다
```

### 3.3 BPE와의 차이점

| 측면 | BPE | WordPiece |
|------|-----|----------|
| **선택 기준** | 빈도 (빠름) | 정보 이득 (의미있음) |
| **공백 처리** | `</w>` | `##` 접두사 |
| **사전 처리** | 필요 없음 | **필수** (단어 경계 인식) |
| **특징** | 효율 중심 | 의미 중심 |

### 3.4 특징
- **선택 기준**: 정보 이득
- **부분단어 표시**: `##` (예: `['play', '##ing']`)
- **사용 모델**: BERT, mBERT, KoBERT
- **어휘 크기**: ~110,000개

---

## 4. SentencePiece - XLM-RoBERTa

### 4.1 핵심 아이디어
**"모든 언어를 동일하게 처리한다. 공백도 그냥 문자다."**

### 4.2 주요 혁신: 메타스페이스(Metaspace)

```
공백을 특수 기호로 표현: ▁ (U+2581, LOWER ONE EIGHTH BLOCK)

"get up"
↓ (메타스페이스 변환)
"get▁up"
↓ (바이트 단위 분해)
['g', 'e', 't', '▁', 'u', 'p']
↓ (BPE 병합)
['get', '▁up']  또는  ['g', 'et', '▁', 'p']
```

### 4.3 처리 순서
1. **입력**: "Hello, world!"
2. **메타스페이스 적용**: "Hello,▁world!"
3. **바이트 분해**: 모든 언어를 통일된 방식으로 처리
4. **BPE 병합**: 빈도 또는 Unigram LM 기준

### 4.4 특징
- **언어 독립적**: 사전 분류 불필요
- **공백 처리**: ▁ 기호로 명시적 표현
- **사용 모델**: XLM-RoBERTa, Llama
- **어휘 크기**: ~250,000개
- **장점**: 다국어 처리에 최적화

---

## 5. 세 알고리즘 비교

### 5.1 요약표

| 항목 | BPE | WordPiece | SentencePiece |
|------|-----|-----------|---------------|
| **선택 기준** | 빈도 | 정보이득 | 빈도 (다국어) |
| **공백 처리** | `</w>` | `[없음]` | `▁` |
| **사전 처리** | 불필요 | **필수** | 불필요 |
| **언어 처리** | 편향 | 편향 | **균등** |
| **사용 모델** | GPT | BERT | XLM, Llama |
| **어휘 크기** | 50K | 110K | 250K |

### 5.2 선택 가이드

**영어 중심**: BPE 또는 WordPiece  
**한글 포함**: WordPiece 또는 SentencePiece  
**다국어**: SentencePiece (**필수**)  

### 5.3 비용 영향

```
같은 의미의 텍스트를 처리할 때:
- BPE:  10개 토큰 → $0.005
- WordPiece: 6개 토큰 → $0.003 (40% 절감)
- SentencePiece: 5개 토큰 → $0.0025 (50% 절감)

장기 사용 시 알고리즘 선택이 비용을 2배 이상 좌우!
```

---

## 6. 핵심 정리

### 6.1 각 알고리즘의 핵심

**BPE (Byte Pair Encoding)**
- "가장 자주 나타나는 쌍을 계속 병합"
- 빠르고 효율적
- 영어에 최적화, 한글 처리 비효율

**WordPiece**
- "정보 이득이 가장 큰 쌍을 선택"
- 의미있는 병합
- 한글 처리 중간 수준
- 사전 토크나이저 필수

**SentencePiece**
- "모든 언어를 동일하게 처리"
- 다국어 최적화
- 공백도 명시적 토큰
- 가장 간단하고 일관성 있음

### 6.2 선택 팁

1. **비용**: SentencePiece가 토큰 효율 우수
2. **성능**: WordPiece가 의미 정보 우수
3. **편의성**: SentencePiece가 가장 간단
4. **다국어**: SentencePiece 필수
5. **한글**: WordPiece 또는 SentencePiece 추천

### 6.3 다음 학습
- 02_practice.ipynb: 실제 코드로 3가지 토크나이저 직접 비교
- 각 토크나이저의 실행 결과와 성능 측정